#### Libraries

In [ ]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling.document_converter import DocumentConverter
from dotenv import load_dotenv
from openai import OpenAI
from utils.tokenizer import OpenAITokenizerWrapper
import logging
import lancedb
import dlt
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector
from typing import List, Optional
import torch
import os
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


##### General settings, parameters and utility functions

In [2]:
# Set up logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

# OpenAI API setup
api_key_str = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key_str)
model = "gpt-4o"


documents_folder = "./documents"
websites_list = "./documents/websites.csv"

allowed_formats=[
    "pdf",
    "docx",
    "html",
    "doc",
    "txt",
]

##### Database for chunks documents

In [ ]:
# Create a LanceDB database
db = lancedb.connect("data/lancedb")


# Get the OpenAI embedding function
func = get_registry().get("openai").create(name="text-embedding-3-large")

class ChunkMetadata(LanceModel):
    """
    Class to hold metadata for each chunk.
    """
    filename: str | None
    page_numbers: List[int] | None
    title: str | None

class Chunks(LanceModel):
    """Class to represent the main schema for the database."""
    text: str = func.SourceField()
    vector: Vector(func.ndims()) = func.VectorField()  # type: ignore
    metadata: ChunkMetadata

##### Documents or webpage conversion

In [ ]:
def convert_document(file_path: str):
    """ Convert a document to a structured format using DocumentConverter."""
    # Initialize the DocumentConverter
    converter = DocumentConverter()
    # Convert the document to a structured format
    result_guide = converter.convert(file_path)
    # Extract the document guide from the conversion result
    document_guide = result_guide.document
    return document_guide

def convert_documents(file_path: str, chunks) -> DocumentConverter:
    """ Convert documents in a directory to a structured format using DocumentConverter."""
    tokenizer = OpenAITokenizerWrapper()  # Load our custom tokenizer for OpenAI
    MAX_TOKENS = 8191  # text-embedding-3-large's maximum context length    

    # Loop through each file in the directory
    for file_name in os.listdir(documents_folder):
        # Check if the file is a document (e.g., .txt or .csv)
        if file_name.endswith(tuple(allowed_formats)):  
            document_converter = convert_document(os.path.join(documents_folder, file_name))
            
            # Apply hybrid chunking   
            chunker = HybridChunker(
                tokenizer=tokenizer,
                max_tokens=MAX_TOKENS,
                merge_peers=True,
            )

            chunk_iter = chunker.chunk(dl_doc=document_converter)
            chunks.extend(chunk_iter)
    return chunks     

def convert_websites(websites_list: str, chunks) -> DocumentConverter:
    """ Convert websites in a list to a structured format using DocumentConverter."""
    tokenizer = OpenAITokenizerWrapper()  # Load our custom tokenizer for OpenAI
    MAX_TOKENS = 8191  # text-embedding-3-large's maximum context length
    df = pd.read_csv(websites_list)
    for index, row in df.iterrows():
        # Extract the URL from the row
        url = row["URL"]
        # Convert the URL to a structured format
        document_converter = convert_document(url)

        # Apply hybrid chunking   
        chunker = HybridChunker(
                tokenizer=tokenizer,
                max_tokens=MAX_TOKENS,
                merge_peers=True,
        )

        chunk_iter = chunker.chunk(dl_doc=document_converter)
        chunks.extend(chunk_iter)
    return chunks

##### Process chunks and save in database

In [ ]:
def process_chunks(chunks):
    """ Process the chunks to extract relevant metadata and text."""
    processed_chunks = [
        {
            "text": chunk.text,
            "metadata": {
                "filename": chunk.meta.origin.filename,
                "page_numbers": [
                    page_no
                    for page_no in sorted(
                        set(
                            prov.page_no
                            for item in chunk.meta.doc_items
                            for prov in item.prov
                        )
                    )
                ]
                or None,
                "title": chunk.meta.headings[0] if chunk.meta.headings else None,
            },
        }
        for chunk in chunks
    ]
 
    # Create the table in LanceDB (No `vector_column` argument needed)
    table = db.create_table(
        "docling",
        schema=Chunks,
        mode="overwrite"
    )

    table.add(processed_chunks)

    return table

##### Load and save documents 

In [ ]:
chunks = [] 
chunks = convert_documents(documents_folder, chunks)
chunks = convert_websites(websites_list, chunks)
table = process_chunks(chunks)

2025-05-05 20:42:23 - INFO - Going to convert document batch...
2025-05-05 20:42:25 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:27 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:28 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:29 - INFO - Processing document ManualOperation_generated.pdf
2025-05-05 20:42:31 - INFO - Finished converting document ManualOperation_generated.pdf in 11.31 sec.
2025-05-05 20:42:31 - INFO - Going to convert document batch...
2025-05-05 20:42:31 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:33 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:33 - INFO - Accelerator device: 'cpu'
2025-05-05 20:42:34 - INFO - Processing document Oil and gas production handbook ed3x0_web.pdf
2025-05-05 20:50:22 - INFO - Finished converting document Oil and gas production handbook ed3x0_web.pdf in 470.47 sec.
2025-05-05 20:50:23 - INFO - Going to convert document batch...
2025-05-05 20:50:23 - INFO - Processing document Oilfield_scale_inhibition
2025-05-

##### Verify table with the vectors documents stored

In [ ]:

# Connect to the database
uri = "data/lancedb"
db = lancedb.connect(uri)
print(table.to_pandas().head)  # Print table contents

# Load the table
table = db.open_table("docling")

# Search the table
result = table.search(query="pdf").limit(5)
result.to_pandas()
print(result.to_pandas().head())  # Print search results

<bound method NDFrame.head of                                                 text  \
0  The document below describes the main anomalie...   
1  Håvard Devold\nOil and gas production handbook...   
2  This handbook has been compiled for readers wi...   
3  Except as otherwise indicated, all materials, ...   
4  Introduction  ...................................   
5  Oil  has  been  used  for  lighting  purposes ...   
6  The  oil and  gas  industry facilities and  sy...   
7  In the past, surface features such  as  tar  s...   
8  This illustration gives an overview of typical...   
9  Onshore  production  is  economically viable f...   

                                              vector  \
0  [-0.01954059, 0.015509242, -0.007534569, -0.01...   
1  [0.026132353, 0.002569735, 0.0035746705, -0.00...   
2  [0.019409806, 0.006987244, -0.010169765, -0.02...   
3  [-0.01256122, 0.021057226, -0.017518442, 0.029...   
4  [0.013994407, 0.012142206, -0.005844723, 0.005...   
5  [-0.020336127,

2025-05-05 20:50:27 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


                                                text  \
0  Except as otherwise indicated, all materials, ...   
1  ABB AS P.O. Box 6359 Etterstad NO-0603 Oslo Te...   
2                                                153   
3  Introduction  ...................................   
4  Håvard Devold\nOil and gas production handbook...   

                                              vector  \
0  [-0.01256122, 0.021057226, -0.017518442, 0.029...   
1  [0.037816044, 0.0135121355, -0.007260531, -0.0...   
2  [0.0028861423, 0.0026285641, -0.0033020678, -0...   
3  [0.013994407, 0.012142206, -0.005844723, 0.005...   
4  [0.026132353, 0.002569735, 0.0035746705, -0.00...   

                                            metadata  _distance  
0  {'filename': 'Oil and gas production handbook ...   1.524704  
1  {'filename': 'Oil and gas production handbook ...   1.563870  
2  {'filename': 'Oil and gas production handbook ...   1.591304  
3  {'filename': 'Oil and gas production handbook ...   1.62769